# Advanced PFC Semiconductor Design Notebook — Review Edition

This notebook extends the earlier advanced notebook with the **Option A** upgrades:

- **Heatmap fail-region overlay**
- **CSV export** for summary/comparison/pass-fail tables
- **Top-level design verdict summary**
- Existing features preserved:
  - compare two designs side by side
  - bridge diode vs sync-bottom comparison
  - Vac vs ambient heatmap
  - automatic pass/fail checks against Tj limits

## Required files in the same folder
- `pfc_loss_model_step3_local.py`
- `pfc_visualization_step4.py`

In [ ]:
from pathlib import Path
import importlib.util
import sys
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BACKEND_FILE = 'pfc_loss_model_step3_local.py'
VIZ_FILE = 'pfc_visualization_step4.py'

for f in [BACKEND_FILE, VIZ_FILE]:
    assert Path(f).exists(), f'Missing required file: {f}'

backend_spec = importlib.util.spec_from_file_location('pfc_backend_adv_nb2', BACKEND_FILE)
backend = importlib.util.module_from_spec(backend_spec)
sys.modules['pfc_backend_adv_nb2'] = backend
backend_spec.loader.exec_module(backend)

viz_spec = importlib.util.spec_from_file_location('pfc_viz_adv_nb2', VIZ_FILE)
viz = importlib.util.module_from_spec(viz_spec)
sys.modules['pfc_viz_adv_nb2'] = viz
viz_spec.loader.exec_module(viz)

print('Loaded backend:', BACKEND_FILE)
print('Loaded visualization module:', VIZ_FILE)

## 1) Design setup

In [ ]:
base_cfg = copy.deepcopy(backend.EXAMPLE_DESIGN)

cfg_a = copy.deepcopy(base_cfg)

cfg_b = copy.deepcopy(base_cfg)
# Example alternate design edits — replace with your real comparison case
cfg_b['mosfet']['k_esw'] = 1.10
cfg_b['thermal']['t_ambient'] = 45.0
cfg_b['bridge']['vf_tco'] = -0.0015

# Helper to create sync-bottom bridge variant from an existing design
# Replace placeholder bottom MOSFET data with your real datasheet values.
def make_sync_bottom_variant(cfg):
    c = copy.deepcopy(cfg)
    bridge = copy.deepcopy(c['bridge'])
    bridge['topology'] = 'sync_bottom'
    bridge.pop('n_parallel', None)
    bridge['n_parallel_top'] = 1
    bridge['n_parallel_bottom'] = 1
    bridge['rdson_bottom_25'] = 0.020
    bridge['rdson_bottom_tj'] = [[25, 125], [1.0, 1.5]]
    bridge['qg_bottom'] = 60e-9
    bridge['vg_bottom'] = 12.0
    bridge['rth_jc_bottom'] = 0.8
    bridge['rth_cs_bottom'] = 0.5
    c['bridge'] = bridge
    return c

cfg_bridge_diode = copy.deepcopy(base_cfg)
cfg_bridge_sync = make_sync_bottom_variant(base_cfg)

cfg_a['bridge']['topology'], cfg_b['bridge']['topology'], cfg_bridge_sync['bridge']['topology']

## 1a) Component intake, confirmation and validation gate

Workflow: the agent extracts what it can from an uploaded datasheet, shows a **confirmation table** (anything it could not find is flagged `NOT AVAILABLE`), the designer confirms/fills the gaps, and `validate_design()` then **gates** the run so the engine never silently substitutes a built-in default for a required parameter.

`source` tells the designer where each value comes from: `datasheet` (agent extracts), `application` (board/design context - not on the datasheet), `choice`, or `provided` (handed in by the upstream tool).

In [ ]:
import importlib.util as _ilu, sys as _sys
_spec = _ilu.spec_from_file_location('pfc_intake', 'pfc_component_intake.py')
intake = _ilu.module_from_spec(_spec); _sys.modules['pfc_intake'] = intake; _spec.loader.exec_module(intake)

# (1) Example of what an agent extracted from an uploaded SiC MOSFET datasheet (partial)
agent_extracted_mosfet = dict(
    tech='sic', rdson_25=0.040, rdson_tj=[[25,125],[1.0,1.4]],
    ciss=1100e-12, qgd=22e-9, vth=4.5, vpl=9.0, eoss_at_v=[[100,400],[2e-6,9e-6]],
)  # qg & rth_jc not found on the sheet; vg/rg/rth_cs are application inputs

# (2) Confirmation table shown to the designer (NOT AVAILABLE = needs attention)
display(intake.confirmation_table(agent_extracted_mosfet, 'mosfet'))
print('Missing required for this MOSFET:', intake.missing_required(agent_extracted_mosfet, 'mosfet'))

# (3) Gate every confirmed design before running the loss model
for _nm, _cfg in [('base_cfg', base_cfg), ('cfg_a', cfg_a), ('cfg_b', cfg_b)]:
    _ok, _issues = intake.validate_design(_cfg)
    print(f'{_nm}: {"OK - cleared to run" if _ok else "BLOCKED - incomplete"}')
    if not _ok:
        display(_issues)

## 2) Common helpers

In [ ]:
def flat(result):
    return backend.flatten_result(result)


def vac_sweep_df(cfg, vac_list=None):
    rows = backend.simulate_vac_sweep(cfg, vac_list=vac_list, return_waveforms=False)
    return pd.DataFrame([backend.flatten_result(r) for r in rows])


def compare_two_designs_vac(cfg1, cfg2, label1='Design A', label2='Design B', vac_list=None):
    df1 = vac_sweep_df(cfg1, vac_list=vac_list).copy()
    df2 = vac_sweep_df(cfg2, vac_list=vac_list).copy()
    df1['Design'] = label1
    df2['Design'] = label2
    return pd.concat([df1, df2], ignore_index=True)


def run_single(cfg, vac):
    sp, mos, dio, br, th = backend.design_from_dict(cfg)
    return backend.flatten_result(backend.simulate_point(vac, sp, mos, dio, br, th, return_waveforms=True))


def evaluate_pass_fail(flat_result, tj_limits=None, negative_other_loss_is_fail=True):
    if tj_limits is None:
        tj_limits = {'Tj_FET': 150.0, 'Tj_DIODE': 150.0, 'Tj_BRIDGE_top': 130.0, 'Tj_BRIDGE_bottom': 130.0}
    failures = []
    for key, limit in tj_limits.items():
        if key in flat_result and flat_result[key] > limit:
            failures.append(f'{key} > {limit}C')
    if negative_other_loss_is_fail and flat_result.get('P_OTHER_implied', 0.0) < -1e-6:
        failures.append('P_OTHER_implied < 0')
    status = 'PASS' if len(failures) == 0 else 'FAIL'
    return status, failures


def annotate_pass_fail(df, tj_limits=None):
    status = []
    reasons = []
    for _, row in df.iterrows():
        s, f = evaluate_pass_fail(row.to_dict(), tj_limits=tj_limits)
        status.append(s)
        reasons.append('; '.join(f))
    out = df.copy()
    out['PASS_FAIL'] = status
    out['FAIL_REASONS'] = reasons
    return out


def build_vac_ambient_heatmap(cfg, vac_list, ambient_list, metric='Tj_FET'):
    z = np.zeros((len(ambient_list), len(vac_list)))
    for i, tamb in enumerate(ambient_list):
        c = copy.deepcopy(cfg)
        c.setdefault('thermal', {})['t_ambient'] = float(tamb)
        rows = backend.simulate_vac_sweep(c, vac_list=vac_list, return_waveforms=False)
        flats = [backend.flatten_result(r) for r in rows]
        for j, fr in enumerate(flats):
            z[i, j] = fr[metric]
    return z


def plot_heatmap(z, vac_list, ambient_list, title, cbar_label, cmap='viridis'):
    fig, ax = plt.subplots(figsize=(8.0, 5.2))
    im = ax.imshow(z, aspect='auto', origin='lower', cmap=cmap,
                   extent=[min(vac_list), max(vac_list), min(ambient_list), max(ambient_list)])
    ax.set_title(title)
    ax.set_xlabel('Vac [Vrms]')
    ax.set_ylabel('Ambient temperature [°C]')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    ax.set_xticks(vac_list)
    ax.set_yticks(ambient_list)
    fig.tight_layout()
    plt.show()


def plot_heatmap_with_fail_overlay(z, vac_list, ambient_list, limit, title, cbar_label, cmap='viridis'):
    fig, ax = plt.subplots(figsize=(8.0, 5.2))
    im = ax.imshow(z, aspect='auto', origin='lower', cmap=cmap,
                   extent=[min(vac_list), max(vac_list), min(ambient_list), max(ambient_list)])
    fail_mask = z > limit
    if np.any(fail_mask):
        ax.contour(vac_list, ambient_list, fail_mask.astype(float), levels=[0.5], colors='red', linewidths=2)
        yy, xx = np.where(fail_mask)
        ax.scatter(np.array(vac_list)[xx], np.array(ambient_list)[yy], color='red', s=25, label='Fail region')
        ax.legend(loc='upper left')
    ax.set_title(title)
    ax.set_xlabel('Vac [Vrms]')
    ax.set_ylabel('Ambient temperature [°C]')
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)
    ax.set_xticks(vac_list)
    ax.set_yticks(ambient_list)
    fig.tight_layout()
    plt.show()


def design_review_summary(df, tj_limits=None):
    checked = annotate_pass_fail(df, tj_limits=tj_limits)
    worst_case = checked.sort_values('P_SEMI_total', ascending=False).iloc[0].to_dict()
    hottest_fet = checked.sort_values('Tj_FET', ascending=False).iloc[0].to_dict()
    hottest_diode = checked.sort_values('Tj_DIODE', ascending=False).iloc[0].to_dict()
    hottest_bridge = checked.sort_values('Tj_BRIDGE_top', ascending=False).iloc[0].to_dict()
    overall = 'PASS' if (checked['PASS_FAIL'] == 'PASS').all() else 'FAIL'
    summary = {
        'overall_status': overall,
        'worst_case_loss_vac': worst_case['Vac'],
        'worst_case_semi_loss': worst_case['P_SEMI_total'],
        'worst_fet_vac': hottest_fet['Vac'],
        'worst_fet_tj': hottest_fet['Tj_FET'],
        'worst_diode_vac': hottest_diode['Vac'],
        'worst_diode_tj': hottest_diode['Tj_DIODE'],
        'worst_bridge_vac': hottest_bridge['Vac'],
        'worst_bridge_tj': hottest_bridge['Tj_BRIDGE_top'],
        'fail_count': int((checked['PASS_FAIL'] == 'FAIL').sum())
    }
    return checked, pd.DataFrame([summary])


def export_tables(base_name, **tables):
    export_paths = {}
    for name, df in tables.items():
        path = Path(f'{base_name}_{name}.csv')
        df.to_csv(path, index=False)
        export_paths[name] = str(path)
    return export_paths

## 2b) Integration hook - operating point supplied by an upstream script

When this notebook becomes part of a larger tool that already computes the PFC operating point, pass the **input voltage, input power, input current, inductance (or ripple), power factor and efficiency** straight into the spec; the engine uses them verbatim instead of re-deriving them.

Line-current precedence: `iin_rms` > `pin/(Vac*PF)` > `Po/(eta*Vac*PF)`. `pct_ripple` or `di_pp_peak` override the inductance `L`.

In [ ]:
# Values the bigger script would hand in (example: Infineon 90 Vac full-load bench point)
upstream = {
    'vac': 90.0,
    'pin': 1278.5,        # input power [W]
    'iin_rms': 14.40,     # input current [A]
    'pf': 0.9996,         # power factor
    'eta': 0.93829,       # efficiency (fraction)
    'pct_ripple': 0.25,   # inductor ripple as fraction of peak (or set 'L' / 'di_pp_peak')
}

cfg_upstream = copy.deepcopy(base_cfg)
cfg_upstream['spec'].update(dict(
    pin=upstream['pin'], iin_rms=upstream['iin_rms'],
    pf=upstream['pf'], eta=upstream['eta'], pct_ripple=upstream['pct_ripple'],
))
for k in ['po_curve', 'eta_curve', 'pf_curve']:   # drop curves so supplied scalars are unambiguous
    cfg_upstream['spec'].pop(k, None)

r_up = run_single(cfg_upstream, upstream['vac'])
pd.Series({k: r_up[k] for k in [
    'Vac','Pin','Iin_rms','PF_in','eta_in_%','ripple_pk_%','L_eff_uH',
    'P_FET_total','P_DIODE_total','P_BRIDGE_total','P_SEMI_total','P_OTHER_implied']})

## 3) Top-level design verdict summary

This gives a quick review summary before diving into detailed plots.

In [ ]:
limits = {
    'Tj_FET': 150.0,
    'Tj_DIODE': 150.0,
    'Tj_BRIDGE_top': 130.0,
    'Tj_BRIDGE_bottom': 130.0,
}

base_vac_df = vac_sweep_df(cfg_a, vac_list=[90, 115, 180, 230, 265])
base_checked_df, review_summary_df = design_review_summary(base_vac_df, tj_limits=limits)
review_summary_df

In [ ]:
if review_summary_df.loc[0, 'overall_status'] == 'PASS':
    print('Design verdict: PASS')
else:
    print('Design verdict: FAIL')
    display(base_checked_df[base_checked_df['PASS_FAIL'] == 'FAIL'][['Vac', 'FAIL_REASONS']])

## 4) Compare two designs side by side

In [ ]:
compare_df = compare_two_designs_vac(cfg_a, cfg_b, label1='Design A', label2='Design B', vac_list=[90, 115, 180, 230, 265])
compare_df[['Design', 'Vac', 'P_FET_total', 'P_DIODE_total', 'P_BRIDGE_total', 'P_SEMI_total', 'Tj_FET', 'Tj_DIODE', 'Tj_BRIDGE_top']]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for design in compare_df['Design'].unique():
    sub = compare_df[compare_df['Design'] == design]
    axs[0].plot(sub['Vac'], sub['P_SEMI_total'], marker='o', label=design)
    axs[1].plot(sub['Vac'], sub['Tj_FET'], marker='o', label=f'{design} - FET')
    axs[1].plot(sub['Vac'], sub['Tj_DIODE'], marker='s', label=f'{design} - Diode')
axs[0].set_title('Semiconductor Total Loss Comparison')
axs[0].set_xlabel('Vac [Vrms]')
axs[0].set_ylabel('P_SEMI_total [W]')
axs[0].grid(True)
axs[0].legend()
axs[1].set_title('Junction Temperature Comparison')
axs[1].set_xlabel('Vac [Vrms]')
axs[1].set_ylabel('Temperature [°C]')
axs[1].grid(True)
axs[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5) Bridge diode vs sync-bottom comparison

In [ ]:
bridge_compare_df = compare_two_designs_vac(
    cfg_bridge_diode,
    cfg_bridge_sync,
    label1='Bridge diode',
    label2='Sync-bottom',
    vac_list=[90, 115, 180, 230, 265]
)
bridge_compare_df[['Design', 'Vac', 'P_BRIDGE_total', 'P_SEMI_total', 'Tj_BRIDGE_top', 'Tj_BRIDGE_bottom', 'Tj_FET']]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for design in bridge_compare_df['Design'].unique():
    sub = bridge_compare_df[bridge_compare_df['Design'] == design]
    axs[0].plot(sub['Vac'], sub['P_BRIDGE_total'], marker='o', label=design)
    axs[1].plot(sub['Vac'], sub['P_SEMI_total'], marker='o', label=design)
axs[0].set_title('Bridge Loss Comparison')
axs[0].set_xlabel('Vac [Vrms]')
axs[0].set_ylabel('P_BRIDGE_total [W]')
axs[0].grid(True)
axs[0].legend()
axs[1].set_title('Total Semiconductor Loss Comparison')
axs[1].set_xlabel('Vac [Vrms]')
axs[1].set_ylabel('P_SEMI_total [W]')
axs[1].grid(True)
axs[1].legend()
plt.tight_layout()
plt.show()

## 6) Vac vs ambient heatmap

In [ ]:
vac_list = [90, 115, 180, 230, 265]
ambient_list = [25, 35, 45, 55, 65]
metric = 'Tj_FET'

z = build_vac_ambient_heatmap(cfg_a, vac_list=vac_list, ambient_list=ambient_list, metric=metric)
plot_heatmap(z, vac_list, ambient_list,
             title=f'{metric} Heatmap vs Vac and Ambient',
             cbar_label=f'{metric} value')

## 7) Heatmap with fail-region overlay

This overlays the fail region for the chosen temperature limit.

In [ ]:
limit_for_metric = 150.0
plot_heatmap_with_fail_overlay(
    z, vac_list, ambient_list,
    limit=limit_for_metric,
    title=f'{metric} Heatmap with Fail Overlay (> {limit_for_metric}C)',
    cbar_label=f'{metric} value'
)

## 8) Automatic pass/fail checks table

In [ ]:
pf_df = annotate_pass_fail(vac_sweep_df(cfg_a, vac_list=[90, 115, 180, 230, 265]), tj_limits=limits)
pf_df[['Vac', 'P_SEMI_total', 'Tj_FET', 'Tj_DIODE', 'Tj_BRIDGE_top', 'Tj_BRIDGE_bottom', 'PASS_FAIL', 'FAIL_REASONS']]

## 9) Side-by-side single-point comparison table

In [ ]:
selected_vac = 90.0
r_a = run_single(cfg_a, selected_vac)
r_b = run_single(cfg_b, selected_vac)

single_compare_df = pd.DataFrame({
    'Metric': ['P_FET_total', 'P_DIODE_total', 'P_BRIDGE_total', 'P_SEMI_total', 'Tj_FET', 'Tj_DIODE', 'Tj_BRIDGE_top', 'P_OTHER_implied'],
    'Design A': [r_a['P_FET_total'], r_a['P_DIODE_total'], r_a['P_BRIDGE_total'], r_a['P_SEMI_total'], r_a['Tj_FET'], r_a['Tj_DIODE'], r_a['Tj_BRIDGE_top'], r_a['P_OTHER_implied']],
    'Design B': [r_b['P_FET_total'], r_b['P_DIODE_total'], r_b['P_BRIDGE_total'], r_b['P_SEMI_total'], r_b['Tj_FET'], r_b['Tj_DIODE'], r_b['Tj_BRIDGE_top'], r_b['P_OTHER_implied']],
})
single_compare_df

## 10) CSV export

This exports the most useful tables for reporting or sharing.

In [ ]:
exported = export_tables(
    'pfc_review',
    design_review_summary=review_summary_df,
    base_pass_fail=pf_df,
    design_compare=compare_df,
    bridge_compare=bridge_compare_df,
    single_point_compare=single_compare_df,
)
exported

## 11) Existing Step 4 plots for one design

In [ ]:
files = viz.build_step4_visuals(cfg_a, selected_vac=90, vac_list=[90, 115, 180, 230, 265], output_prefix='advanced_review_step4', show=False)
files

In [ ]:
from IPython.display import Image, display
display(Image(filename=files['losses_vs_vac']))
display(Image(filename=files['temperatures_vs_vac']))
display(Image(filename=files['loss_breakdown']))
display(Image(filename=files['waveforms']))

## 12) Next extensions

Good next additions:
- fail-region overlays for **multiple** metrics at once
- CSV + Excel export in one click
- compare **two different device sets** side by side
- add a small design-review front page with worst-case operating point snapshots
- move the same workflow into a **Streamlit UI** once the notebook flow is finalized